# 01 · Dataset scope

Phase 1 of the M2.1 checkpoint: load the Kenya Law Corpus, see what is actually in it, filter to case law, and freeze a 400-record slice that every later phase uses.

The dataset is loaded at a fixed revision because the Hugging Face card says the corpus is an actively maintained scrape. Without pinning, the same seed could give a different sample next week.

In [1]:
import pandas as pd
from datasets import load_dataset

DATASET = "Daudipdg/kenya-law-corpus"
REVISION = "8baca7e3b15afe9cb427b5adf8c851f889cc90fd"  # HF commit, last modified 2026-07-01
SEED = 42
SLICE_SIZE = 400

## 1. Load and inspect the raw structure

In [2]:
ds = load_dataset(DATASET, revision=REVISION)
ds

DatasetDict({
    train: Dataset({
        features: ['chunk_id', 'text', 'metadata'],
        num_rows: 63603
    })
})

In [3]:
ds["train"].features

{'chunk_id': Value('string'),
 'text': Value('string'),
 'metadata': Json(decode=True)}

There is one split (`train`) with three columns. `title`, `url`, `type` and the rest are **not** top-level columns: they are nested inside `metadata`. Here is a single record:

In [4]:
ds["train"][0]

{'chunk_id': 'https://new.kenyalaw.org/akn/ke/judgment/kemc/2025/94/eng@2025-05-15##chunk0',
 'text': 'Republic v Chumba (Traffic Case E827 of 2025) [2025] KEMC 94 (KLR) (15 May 2025) (Sentence) Neutral citation: [2025] KEMC 94 (KLR) Republic of Kenya In the Nakuru Law Courts Traffic Case E827 of 2025 PA Ndege, SPM May 15, 2025 Between Republic Prosecution and Denis Cheruiyot Chumba Accused Sentence 1. The accused person herein, Denis Cheruiyot Chumba, has been convicted upon own plea of guilty of the offence of operating a motor vehicle on a public road for hire or reward without the requisite license issued by the authority contrary to Section 26(1) as read with Section 26(7) of the National Transport and Safety Authority Act . He has admitted that on 9/05/2025 at about 1100hours along Nakuru-Nyahururu road at Kagoto area within Nakuru County, being the driver of a motor vehicle Reg No. K',
 'metadata': {'title': 'Republic v Chumba (Traffic Case E827\xa0of\xa02025) [2025]\xa0KEMC\xa0

## 2. Flatten the metadata into columns

In [5]:
raw = ds["train"].to_pandas()
meta = pd.json_normalize(raw["metadata"])
df = pd.concat([raw.drop(columns="metadata"), meta], axis=1)

print(df.shape)
df.head()

(63603, 10)


,chunk_id,text,title,url,type,chunk_index,total_chunks,citation,court,date
0,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Chumba (Traffic Case E827 of 2025) ...,Republic v Chumba (Traffic Case E827 of 2025) ...,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case_law,0,4,,,"May 15, 2025"
1,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,ed that on 9/05/2025 at about 1100hours along ...,Republic v Chumba (Traffic Case E827 of 2025) ...,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case_law,1,4,,,"May 15, 2025"
2,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,chool bus without road service license contrar...,Republic v Chumba (Traffic Case E827 of 2025) ...,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case_law,2,4,,,"May 15, 2025"
3,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,IGNED AND DELIVERED AT NAKURU IN OPEN COURT TH...,Republic v Chumba (Traffic Case E827 of 2025) ...,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case_law,3,4,,,"May 15, 2025"
4,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Republic v Joshua (Criminal Case 765 of 2018) ...,Republic v Joshua (Criminal Case 765 of 2018) ...,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,case_law,0,26,,,"July 10, 2025"


In [6]:
df.isna().sum()

chunk_id            0
text                0
title               0
url                 0
type                0
chunk_index         0
total_chunks        0
citation        46250
court           46250
date            46250
dtype: int64

`citation`, `court` and `date` are missing on 46,250 rows. The next cell shows that this matches the number of legislation rows: those fields simply don't exist for legislation.

In [7]:
df["type"].value_counts()

type
legislation    46250
case_law       17353
Name: count, dtype: int64

## 3. Filter to case law

In [8]:
case_law = df[df["type"] == "case_law"].reset_index(drop=True)

print("case_law chunks:   ", len(case_law))
print("distinct judgments:", case_law["url"].nunique())

case_law chunks:    17353
distinct judgments: 990


Each row is a **chunk** of a judgment, not a whole judgment. Some quick checks on what those chunks look like:

In [9]:
case_law["text"].str.len().describe()

count    17353.000000
mean       773.106725
std        114.517224
min          1.000000
25%        800.000000
50%        800.000000
75%        800.000000
max        800.000000
Name: text, dtype: float64

In [10]:
# Empty strings, not NaN: the fields exist for case law but were never filled
(case_law[["citation", "court", "date"]] == "").sum()

citation    17353
court       17353
date          727
dtype: int64

In [11]:
# The court code is in the URL path, e.g. /judgment/kemc/ = Magistrates' Courts
case_law["url"].str.extract(r"/judgment/([a-z]+)/")[0].value_counts()

0
kemc      11897
kehc       1637
keelc      1541
keca        750
kekc        726
keelrc      648
kesc        154
Name: count, dtype: int64

What this shows:
- Chunks are capped at **800 characters** and most are exactly 800, so the text is cut at a fixed length, often mid-sentence or mid-word.
- `citation` and `court` are empty on every case-law row, and `date` is empty on 727 rows. The court can still be recovered from the URL.
- Magistrates' Courts (`kemc`) make up about 69% of case-law chunks, so a random sample will be skewed the same way.

## 4. Freeze the 400-record slice

Sampling is a simple random sample of chunks with a fixed seed. Duplicates are deliberately **not** removed here: that is a cleaning step, and it is handled in Phase 3 so the before/after counts are visible there.

In [12]:
slice_df = case_law.sample(n=SLICE_SIZE, random_state=SEED).reset_index(drop=True)

print(slice_df.shape)
print("distinct judgments in slice:", slice_df["url"].nunique())
slice_df[["chunk_id", "title"]].head()

(400, 10)
distinct judgments in slice: 272


,chunk_id,title
0,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Sikale v Kichachu & 2 others (Environment & La...
1,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,Ombati v Kariuki & another (Civil Case E486 of...
2,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,In re Estate of Hannah Wanjiru Kamau (Deceased...
3,https://new.kenyalaw.org/akn/ke/judgment/kehc/...,Karani v Republic (Criminal Appeal E045 of 202...
4,https://new.kenyalaw.org/akn/ke/judgment/kekc/...,In re Estate of Late Mude Keko (Deceased) (Suc...


Only the `chunk_id`s are saved to the repo, not the judgment text. The dataset card says the licence is *"not yet finalized"*, so redistributing the text isn't clearly allowed. The IDs plus the pinned revision are enough to rebuild the same slice exactly.

In [13]:
slice_df[["chunk_id"]].to_csv("../data/slice_chunk_ids.csv", index=False)

## 5. Check that the frozen slice can be rebuilt

In [14]:
ids = pd.read_csv("../data/slice_chunk_ids.csv")["chunk_id"]
rebuilt = case_law[case_law["chunk_id"].isin(ids)]

print("rows rebuilt:", len(rebuilt))
print("same chunks: ", set(rebuilt["chunk_id"]) == set(slice_df["chunk_id"]))

rows rebuilt: 400
same chunks:  True


`chunk_id` is unique within case law (the 1,291 repeated `chunk_id`s in the full corpus are all legislation), so rebuilding by ID returns exactly 400 rows.

Later notebooks load the slice this way: the same `load_dataset` call at `REVISION`, flatten, filter to `case_law`, then keep the IDs in `data/slice_chunk_ids.csv`.